# Procesamiento de fuentes RSS

Este notebook implementa el procesamiento de los contenidos RSS almacenados
previamente en la capa RAW.

## Objetivos

1. Obtener los registros pendientes de procesamiento desde `rss_raw`.
2. Interpretar el contenido RSS y extraer las noticias individuales.
3. Normalizar los campos obtenidos desde las distintas fuentes.
4. Evaluar la relevancia de las noticias según las palabras clave configuradas.
5. Almacenar las noticias resultantes en `rss_clean`.
6. Marcar los registros RAW procesados para evitar reprocesamientos
   innecesarios.

La información almacenada en `rss_raw` se mantiene sin modificaciones,
preservando los datos originales obtenidos desde cada fuente.

In [40]:
import os
from pathlib import Path

import feedparser
import pandas as pd
import psycopg
from dotenv import load_dotenv
from bs4 import BeautifulSoup
import re

import hashlib

In [2]:
PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

True

In [3]:
def get_database_connection():
    """Crea una conexión con la base de datos PostgreSQL configurada.

    Returns:
        psycopg.Connection: Conexión activa a la base de datos PostgreSQL.
    """
    return psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )

In [6]:
def get_pending_rss_raw():
    """Obtiene los contenidos RSS pendientes de procesamiento.

    Returns:
        list[tuple]: Registros RAW que todavía no fueron procesados.
    """
    select_query = """
        SELECT
            rss_raw_id,
            fk_rss_source_id,
            raw_content
        FROM rss_raw
        WHERE is_processed = FALSE
        ORDER BY rss_raw_id;
    """

    with get_database_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(select_query)
            return cursor.fetchall()

In [7]:
def get_active_keywords():
    """Obtiene las palabras clave activas configuradas en PostgreSQL.

    Returns:
        list[tuple]: Identificador y texto de las palabras clave activas.
    """
    select_query = """
        SELECT
            keyword_id,
            keyword
        FROM keywords
        WHERE is_active = TRUE
        ORDER BY keyword_id;
    """

    with get_database_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(select_query)
            return cursor.fetchall()

In [10]:
def parse_rss_entries(raw_records):
    """Extrae las entradas individuales de los contenidos RSS.

    Args:
        raw_records (list[tuple]): Registros RAW pendientes de procesamiento.

    Returns:
        list[dict]: Entradas RSS individuales junto con su trazabilidad.
    """
    entries = []

    for rss_raw_id, rss_source_id, raw_content in raw_records:
        feed = feedparser.parse(raw_content)

        for entry in feed.entries:
            entries.append(
                {
                    "rss_raw_id": rss_raw_id,
                    "rss_source_id": rss_source_id,
                    "title": entry.get("title"),
                    "description": entry.get("description"),
                    "link": entry.get("link"),
                    "author": entry.get("author"),
                    "published": entry.get("published"),
                    "has_content": bool(entry.get("content")),
                }
            )

    return entries

In [8]:
pending_raw = get_pending_rss_raw()

print(f"Registros RAW pendientes: {len(pending_raw)}")

Registros RAW pendientes: 7


In [9]:
active_keywords = get_active_keywords()

print(f"Palabras clave activas: {len(active_keywords)}")
active_keywords

Palabras clave activas: 6


[(1, 'inteligencia artificial'),
 (2, 'IA generativa'),
 (3, 'machine learning'),
 (4, 'aprendizaje automático'),
 (5, 'modelo de lenguaje'),
 (6, 'LLM')]

In [11]:
parsed_entries = parse_rss_entries(pending_raw)

print(f"Noticias extraídas: {len(parsed_entries)}")

Noticias extraídas: 186


In [12]:
df_parsed_entries = pd.DataFrame(parsed_entries)

df_parsed_entries.head(20)

,rss_raw_id,rss_source_id,title,description,link,author,published,has_content
0,1,1,Guía para obtener 15 GB de espacio gratis en G...,"Videos, respaldos antiguos e instaladores suel...",https://www.infobae.com/tecno/2026/08/27/guia-...,Santiago Neira,"Thu, 27 Aug 2026 16:55:50 +0000",True
1,1,1,Epic Games Store regala un nuevo juego de terr...,La tienda suma un indie de supervivencia y exp...,https://www.infobae.com/tecno/2026/08/27/epic-...,Santiago Neira,"Thu, 27 Aug 2026 16:28:47 +0000",True
2,1,1,Bill Gates advierte sobre la IA: “será el mayo...,"Según el fundador de Microsoft, esta tecnologí...",https://www.infobae.com/tecno/2026/08/27/bill-...,Dylan Escobar Ruiz,"Thu, 27 Aug 2026 16:11:37 +0000",True
3,1,1,WhatsApp refuerza la seguridad de las cuentas ...,La nueva función busca combatir las tácticas d...,https://www.infobae.com/tecno/2026/08/27/whats...,Pedro Noriega,"Thu, 27 Aug 2026 16:10:29 +0000",True
4,1,1,Gratis en Steam: el juego que combina superviv...,El juego mezcla supervivencia con dinosaurios ...,https://www.infobae.com/tecno/2026/08/27/grati...,Santiago Neira,"Thu, 27 Aug 2026 15:41:10 +0000",True
5,1,1,Truco para ampliar la cobertura del internet e...,"El dispositivo, convertido en un repetidor WiF...",https://www.infobae.com/tecno/2026/08/27/truco...,Dylan Escobar Ruiz,"Thu, 27 Aug 2026 15:39:22 +0000",True
6,1,1,Time reveló las 100 personalidades más influye...,Veronyka Gimenes es la única latinoamericana q...,https://www.infobae.com/tecno/2026/08/27/time-...,Isabela Durán San Juan,"Thu, 27 Aug 2026 15:37:49 +0000",True
7,1,1,Un informe reveló cómo 700 agentes de IA de Op...,"La empresa, conocida por crear ChatGPT, recono...",https://www.infobae.com/tecno/2026/08/27/un-in...,Catalina Puppo Ávila,"Thu, 27 Aug 2026 15:30:29 +0000",True
8,1,1,Qué presentaría Apple en su evento del 9 de se...,Apple daría a conocer sus nuevos modelos de iP...,https://www.infobae.com/tecno/2026/08/27/que-p...,Pedro Noriega,"Thu, 27 Aug 2026 15:29:16 +0000",True
9,1,1,A qué hora se estrena el avance extendido de G...,Rockstar Games ofrecerá un nuevo vistazo al es...,https://www.infobae.com/tecno/2026/08/27/a-que...,Pedro Noriega,"Thu, 27 Aug 2026 15:03:11 +0000",True


In [13]:
df_parsed_entries.groupby("rss_source_id").agg(
    news_count=("link", "count"),
    descriptions=("description", lambda values: values.notna().sum()),
    authors=("author", lambda values: values.notna().sum()),
    published_dates=("published", lambda values: values.notna().sum()),
    contents=("has_content", "sum"),
)

,news_count,descriptions,authors,published_dates,contents
rss_source_id,,,,,
1,100,100,98,100,100
2,10,10,10,10,0
3,12,12,12,12,12
4,12,12,12,12,12
5,12,12,12,12,12
6,15,15,15,15,0
7,25,25,25,25,15


## Normalización de noticias

Las entradas obtenidas desde las diferentes fuentes RSS pueden presentar
variaciones en sus campos y estructura.

En esta etapa se normalizan las noticias a una representación común antes
de almacenarlas en `rss_news`. Cuando una fuente no proporciona contenido
completo, se utiliza la descripción como texto disponible para el
procesamiento posterior.

In [15]:
def clean_html(value):
    """Elimina etiquetas HTML y normaliza espacios de un texto.

    Args:
        value (str | None): Texto que puede contener etiquetas HTML.

    Returns:
        str | None: Texto limpio o None cuando no existe contenido.
    """
    if not value:
        return None

    soup = BeautifulSoup(value, "html.parser")
    text = soup.get_text(separator=" ", strip=True)

    return " ".join(text.split())

In [16]:
from email.utils import parsedate_to_datetime


def normalize_rss_entry(entry):
    """Normaliza una entrada RSS a la estructura utilizada por rss_news.

    Args:
        entry (dict): Entrada obtenida mediante feedparser.

    Returns:
        dict: Noticia con estructura normalizada.
    """
    description = clean_html(entry.get("description"))

    content_items = entry.get("content", [])
    content = None

    if content_items:
        content = clean_html(content_items[0].get("value"))

    if not content:
        content = description

    published = entry.get("published")
    published_at = (
        parsedate_to_datetime(published)
        if published
        else None
    )

    return {
        "title": clean_html(entry.get("title")),
        "description": description,
        "content": content,
        "url": entry.get("link"),
        "author": entry.get("author"),
        "published_at": published_at,
    }

In [17]:
def clean_html(value):
    """Elimina etiquetas HTML y normaliza los espacios de un texto.

    Args:
        value (str | None): Texto que puede contener etiquetas HTML.

    Returns:
        str | None: Texto normalizado o None si no existe contenido.
    """
    if not value:
        return None

    soup = BeautifulSoup(value, "html.parser")
    text = soup.get_text(separator=" ", strip=True)

    return " ".join(text.split())

In [18]:
example_html = "<p>Noticia sobre <strong>inteligencia artificial</strong>.</p>"

clean_html(example_html)

'Noticia sobre inteligencia artificial .'

In [19]:
def parse_published_date(value):
    """Convierte una fecha RSS a un objeto datetime.

    Args:
        value (str | None): Fecha publicada por la fuente RSS.

    Returns:
        datetime | None: Fecha normalizada o None si no está disponible.
    """
    if not value:
        return None

    try:
        return parsedate_to_datetime(value)
    except (TypeError, ValueError):
        return None

In [20]:
def normalize_rss_entry(entry, rss_raw_id, rss_source_id):
    """Normaliza una entrada RSS a la estructura de rss_news.

    Args:
        entry (feedparser.FeedParserDict): Entrada RSS original.
        rss_raw_id (int): Identificador del registro RAW de origen.
        rss_source_id (int): Identificador de la fuente RSS.

    Returns:
        dict: Noticia normalizada.
    """
    description = clean_html(entry.get("description"))

    content_items = entry.get("content", [])
    content = None

    if content_items:
        content = clean_html(content_items[0].get("value"))

    if not content:
        content = description

    return {
        "rss_raw_id": rss_raw_id,
        "rss_source_id": rss_source_id,
        "title": clean_html(entry.get("title")),
        "description": description,
        "content": content,
        "url": entry.get("link"),
        "author": clean_html(entry.get("author")),
        "published_at": parse_published_date(
            entry.get("published")
        ),
    }

In [21]:
def normalize_rss_entries(raw_records):
    """Extrae y normaliza las noticias de los registros RSS pendientes.

    Args:
        raw_records (list[tuple]): Registros RAW pendientes de procesamiento.

    Returns:
        list[dict]: Noticias individuales normalizadas.
    """
    normalized_entries = []

    for rss_raw_id, rss_source_id, raw_content in raw_records:
        feed = feedparser.parse(raw_content)

        for entry in feed.entries:
            normalized_entry = normalize_rss_entry(
                entry=entry,
                rss_raw_id=rss_raw_id,
                rss_source_id=rss_source_id,
            )
            normalized_entries.append(normalized_entry)

    return normalized_entries

In [22]:
normalized_entries = normalize_rss_entries(pending_raw)

print(f"Noticias normalizadas: {len(normalized_entries)}")

Noticias normalizadas: 186


In [23]:
df_normalized_entries = pd.DataFrame(normalized_entries)

df_normalized_entries.head(20)

,rss_raw_id,rss_source_id,title,description,content,url,author,published_at
0,1,1,Guía para obtener 15 GB de espacio gratis en G...,"Videos, respaldos antiguos e instaladores suel...",Disponer de espacio de almacenamiento en la nu...,https://www.infobae.com/tecno/2026/08/27/guia-...,Santiago Neira,2026-08-27 16:55:50+00:00
1,1,1,Epic Games Store regala un nuevo juego de terr...,La tienda suma un indie de supervivencia y exp...,Epic Games Store suma una nueva propuesta a su...,https://www.infobae.com/tecno/2026/08/27/epic-...,Santiago Neira,2026-08-27 16:28:47+00:00
2,1,1,Bill Gates advierte sobre la IA: “será el mayo...,"Según el fundador de Microsoft, esta tecnologí...",El empresario y filántropo Bill Gates lanzó un...,https://www.infobae.com/tecno/2026/08/27/bill-...,Dylan Escobar Ruiz,2026-08-27 16:11:37+00:00
3,1,1,WhatsApp refuerza la seguridad de las cuentas ...,La nueva función busca combatir las tácticas d...,WhatsApp anunció nuevas herramientas de seguri...,https://www.infobae.com/tecno/2026/08/27/whats...,Pedro Noriega,2026-08-27 16:10:29+00:00
4,1,1,Gratis en Steam: el juego que combina superviv...,El juego mezcla supervivencia con dinosaurios ...,"PixARK, el juego que fusiona la supervivencia ...",https://www.infobae.com/tecno/2026/08/27/grati...,Santiago Neira,2026-08-27 15:41:10+00:00
5,1,1,Truco para ampliar la cobertura del internet e...,"El dispositivo, convertido en un repetidor WiF...",Algunos desconocen que es posible convertir un...,https://www.infobae.com/tecno/2026/08/27/truco...,Dylan Escobar Ruiz,2026-08-27 15:39:22+00:00
6,1,1,Time reveló las 100 personalidades más influye...,Veronyka Gimenes es la única latinoamericana q...,La revista Time ha realizado una lista de las ...,https://www.infobae.com/tecno/2026/08/27/time-...,Isabela Durán San Juan,2026-08-27 15:37:49+00:00
7,1,1,Un informe reveló cómo 700 agentes de IA de Op...,"La empresa, conocida por crear ChatGPT, recono...",OpenAI publicó un extenso informe con los resu...,https://www.infobae.com/tecno/2026/08/27/un-in...,Catalina Puppo Ávila,2026-08-27 15:30:29+00:00
8,1,1,Qué presentaría Apple en su evento del 9 de se...,Apple daría a conocer sus nuevos modelos de iP...,Apple celebrará un nuevo evento el próximo 9 d...,https://www.infobae.com/tecno/2026/08/27/que-p...,Pedro Noriega,2026-08-27 15:29:16+00:00
9,1,1,A qué hora se estrena el avance extendido de G...,Rockstar Games ofrecerá un nuevo vistazo al es...,Rockstar Games lanzará este jueves 27 de agost...,https://www.infobae.com/tecno/2026/08/27/a-que...,Pedro Noriega,2026-08-27 15:03:11+00:00


In [24]:
df_normalized_entries.isna().sum()

rss_raw_id        0
rss_source_id     0
title             0
description       2
content           0
url               0
author           15
published_at      0
dtype: int64

## Detección de palabras clave

Las noticias normalizadas se analizan utilizando las palabras clave activas
configuradas en la base de datos.

La búsqueda se realiza sobre el título, la descripción y el contenido de
cada noticia. Una misma noticia puede coincidir con múltiples palabras clave.

Esta etapa permite identificar las noticias relacionadas con el dominio
temático de interés sin acoplar los criterios de búsqueda al código.

In [25]:
import unicodedata


def normalize_text(value):
    """Normaliza un texto para facilitar comparaciones.

    Convierte el texto a minúsculas y elimina marcas diacríticas para
    realizar comparaciones independientes de mayúsculas y acentos.

    Args:
        value (str | None): Texto que se desea normalizar.

    Returns:
        str: Texto normalizado.
    """
    if not value:
        return ""

    value = value.lower()

    return "".join(
        character
        for character in unicodedata.normalize("NFD", value)
        if unicodedata.category(character) != "Mn"
    )

In [34]:
def detect_keywords(news, keywords):
    """Detecta palabras clave completas presentes en una noticia.

    La búsqueda se realiza sobre el título y el contenido de la noticia.
    Las coincidencias respetan los límites de las palabras para evitar
    coincidencias parciales no deseadas.

    Args:
        news (dict): Noticia normalizada.
        keywords (list[tuple]): Palabras clave activas configuradas.

    Returns:
        list[dict]: Palabras clave detectadas en la noticia.
    """
    searchable_text = " ".join(
        [
            news.get("title") or "",
            news.get("content") or "",
        ]
    )

    searchable_text = normalize_text(searchable_text)

    matches = []

    for keyword_id, keyword in keywords:
        normalized_keyword = normalize_text(keyword)

        pattern = rf"\b{re.escape(normalized_keyword)}\b"

        if re.search(pattern, searchable_text):
            matches.append(
                {
                    "keyword_id": keyword_id,
                    "keyword": keyword,
                }
            )

    return matches

In [35]:
keyword_results = []

for news_index, news in enumerate(normalized_entries):
    matches = detect_keywords(
        news=news,
        keywords=active_keywords,
    )

    for match in matches:
        keyword_results.append(
            {
                "news_index": news_index,
                "rss_source_id": news["rss_source_id"],
                "title": news["title"],
                "keyword_id": match["keyword_id"],
                "keyword": match["keyword"],
            }
        )

In [36]:
df_keyword_results = pd.DataFrame(keyword_results)

print(
    f"Coincidencias noticia-keyword: "
    f"{len(df_keyword_results)}"
)

df_keyword_results.head(20)

Coincidencias noticia-keyword: 117


,news_index,rss_source_id,title,keyword_id,keyword
0,2,1,Bill Gates advierte sobre la IA: “será el mayo...,1,inteligencia artificial
1,2,1,Bill Gates advierte sobre la IA: “será el mayo...,7,IA
2,6,1,Time reveló las 100 personalidades más influye...,1,inteligencia artificial
3,6,1,Time reveló las 100 personalidades más influye...,7,IA
4,7,1,Un informe reveló cómo 700 agentes de IA de Op...,1,inteligencia artificial
5,7,1,Un informe reveló cómo 700 agentes de IA de Op...,7,IA
6,11,1,El costo de Bitcoin para este jueves,1,inteligencia artificial
7,11,1,El costo de Bitcoin para este jueves,7,IA
8,12,1,Bill Gates pide reservar empleos solo para hum...,1,inteligencia artificial
9,12,1,Bill Gates pide reservar empleos solo para hum...,7,IA


In [37]:
matched_news = df_keyword_results["news_index"].nunique()

print(f"Noticias relacionadas con IA: {matched_news}")
print(f"Noticias analizadas: {len(normalized_entries)}")

Noticias relacionadas con IA: 70
Noticias analizadas: 186


In [38]:
df_keyword_results.groupby("keyword").agg(
    news_count=("news_index", "nunique")
).sort_values(
    "news_count",
    ascending=False,
)

,news_count
keyword,
inteligencia artificial,57
IA,53
IA generativa,4
aprendizaje automático,3


## Persistencia de noticias procesadas

Las noticias normalizadas se almacenan en `rss_news`, conservando la
trazabilidad hacia la fuente y el registro RAW del cual fueron obtenidas.

Posteriormente, las coincidencias detectadas se almacenan en
`rss_news_keywords`.

Una vez procesadas correctamente todas las noticias de un registro RAW,
este se marca como procesado.

In [39]:
def generate_content_hash(content):
    """Genera un hash SHA-256 para el contenido de una noticia.

    Args:
        content (str): Contenido normalizado de la noticia.

    Returns:
        str: Hash SHA-256 representado en formato hexadecimal.
    """
    return hashlib.sha256(
        content.encode("utf-8")
    ).hexdigest()

In [41]:
def save_processed_news(news_entries, keyword_matches):
    """Persiste noticias, relaciones con keywords y actualiza los RAW.

    La operación se ejecuta dentro de una única transacción para garantizar
    que los registros RAW solo sean marcados como procesados cuando todas
    sus noticias y relaciones se hayan almacenado correctamente.

    Args:
        news_entries (list[dict]): Noticias normalizadas.
        keyword_matches (list[dict]): Coincidencias noticia-keyword.

    Returns:
        tuple[int, int]: Cantidad de noticias procesadas y relaciones
        noticia-keyword procesadas.
    """
    news_query = """
        INSERT INTO rss_news (
            fk_rss_source_id,
            fk_rss_raw_id,
            title,
            description,
            content,
            url,
            author,
            published_at,
            content_hash
        )
        VALUES (
            %s, %s, %s, %s, %s, %s, %s, %s, %s
        )
        ON CONFLICT (url)
        DO UPDATE SET
            updated_at = CURRENT_TIMESTAMP
        RETURNING rss_news_id;
    """

    keyword_query = """
        INSERT INTO rss_news_keywords (
            fk_rss_news_id,
            fk_keyword_id
        )
        VALUES (%s, %s)
        ON CONFLICT (
            fk_rss_news_id,
            fk_keyword_id
        )
        DO NOTHING;
    """

    raw_query = """
        UPDATE rss_raw
        SET is_processed = TRUE
        WHERE rss_raw_id = ANY(%s);
    """

    matches_by_news = {}

    for match in keyword_matches:
        news_index = match["news_index"]
        matches_by_news.setdefault(news_index, []).append(
            match["keyword_id"]
        )

    processed_news = 0
    processed_relations = 0

    with get_database_connection() as connection:
        with connection.cursor() as cursor:
            for news_index, news in enumerate(news_entries):
                content_hash = generate_content_hash(
                    news["content"]
                )

                cursor.execute(
                    news_query,
                    (
                        news["rss_source_id"],
                        news["rss_raw_id"],
                        news["title"],
                        news["description"],
                        news["content"],
                        news["url"],
                        news["author"],
                        news["published_at"],
                        content_hash,
                    ),
                )

                rss_news_id = cursor.fetchone()[0]
                processed_news += 1

                for keyword_id in matches_by_news.get(
                    news_index,
                    [],
                ):
                    cursor.execute(
                        keyword_query,
                        (
                            rss_news_id,
                            keyword_id,
                        ),
                    )

                    if cursor.rowcount == 1:
                        processed_relations += 1

            raw_ids = list(
                {
                    news["rss_raw_id"]
                    for news in news_entries
                }
            )

            cursor.execute(
                raw_query,
                (raw_ids,),
            )

        connection.commit()

    return processed_news, processed_relations

In [42]:
processed_news, processed_relations = save_processed_news(
    news_entries=normalized_entries,
    keyword_matches=keyword_results,
)

print(f"Noticias procesadas: {processed_news}")
print(
    f"Relaciones noticia-keyword insertadas: "
    f"{processed_relations}"
)

Noticias procesadas: 186
Relaciones noticia-keyword insertadas: 117
